In [0]:
# =============================================================================
# Notebook  : 02b_map_10_account_events_mapped
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_10_account_events_mapped
# Spec Ref  : §1.6 Event Aggregation — Account-Level (mk_account_events_mapped)
# Purpose   : Aggregate events per account × meta_event × event_day.
#             Combines IDENTIFIED events (anonymous_activity=false)
#             and ANONYMOUS events (anonymous_activity=true).
#             This gives scoring models both signals per account.
#
# Identified: contactid is known → roll up through contacts_to_accounts
# Anonymous : contactid IS NULL → match event to account by domain
#
# Incremental MERGE — only reprocesses events since last aggregated day.
#
# Run after : map_04 (mapped_events), map_05 (contacts_to_accounts)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
print(f"=== Map 10: mk_account_events_mapped (incremental MERGE) ===  customer={customer_id}")

In [0]:
# CELL 3
create_map_table(
    f"{sv}.mk_account_events_mapped",
    """
        mk_accountid_domain  STRING  NOT NULL,
        meta_event           STRING  NOT NULL,
        event_day            DATE    NOT NULL,
        occurrences          BIGINT,
        anonymous_activity   BOOLEAN,
        tenant               BIGINT
    """,
    cluster_by="mk_accountid_domain, meta_event, event_day"
)

In [0]:
# CELL 4 -- Incremental watermark
last_day = spark.sql(f"""
    SELECT COALESCE(MAX(event_day), DATE('1900-01-01'))
    FROM {sv}.mk_account_events_mapped
""").collect()[0][0]
print(f"  Last aggregated day: {last_day}")

# CELL 5 -- Build delta: identified + anonymous events
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW account_events_delta AS

    -- IDENTIFIED: contact is known → roll up via contacts_to_accounts
    SELECT
        c2a.account_id                            AS mk_accountid_domain,
        e.meta_event,
        CAST(e.event_timestamp AS DATE)           AS event_day,
        COUNT(*)                                  AS occurrences,
        false                                     AS anonymous_activity,
        e.tenant
    FROM {sv}.mapped_events e
    INNER JOIN {sv}.contacts_to_accounts c2a
        ON e.contactid = c2a.contact_id
    WHERE e.contactid IS NOT NULL
      AND CAST(e.event_timestamp AS DATE) >= '{last_day}'
    GROUP BY
        c2a.account_id,
        e.meta_event,
        CAST(e.event_timestamp AS DATE),
        e.tenant

    UNION ALL

    -- ANONYMOUS: no contact resolved → match to account by domain
    SELECT
        a.id                                      AS mk_accountid_domain,
        e.meta_event,
        CAST(e.event_timestamp AS DATE)           AS event_day,
        COUNT(*)                                  AS occurrences,
        true                                      AS anonymous_activity,
        e.tenant
    FROM {sv}.mapped_events e
    INNER JOIN {sv}.accounts_all a
        ON LOWER(e.domain) = LOWER(a.domain)
    WHERE e.contactid IS NULL
      AND e.domain IS NOT NULL
      AND TRIM(e.domain) != ''
      AND CAST(e.event_timestamp AS DATE) >= '{last_day}'
    GROUP BY
        a.id,
        e.meta_event,
        CAST(e.event_timestamp AS DATE),
        e.tenant
""")

# Add tenant column if missing (table may pre-date DDL update)
existing_cols = [c.name for c in spark.table(f"{sv}.mk_account_events_mapped").schema]
if "tenant" not in existing_cols:
    spark.sql(f"ALTER TABLE {sv}.mk_account_events_mapped ADD COLUMN tenant BIGINT")

# CELL 6 - FIXED
spark.sql(f"""
    MERGE INTO {sv}.mk_account_events_mapped AS target
    USING account_events_delta AS source
    ON  target.mk_accountid_domain = source.mk_accountid_domain
    AND target.meta_event          = source.meta_event
    AND target.event_day           = source.event_day
    AND target.anonymous_activity  = source.anonymous_activity
    AND target.tenant              = source.tenant
    WHEN MATCHED THEN
        UPDATE SET
            target.occurrences = source.occurrences
    WHEN NOT MATCHED THEN
        INSERT (mk_accountid_domain, meta_event, event_day,
                occurrences, anonymous_activity, tenant)
        VALUES (source.mk_accountid_domain, source.meta_event,
                source.event_day, source.occurrences,
                source.anonymous_activity, source.tenant)
""")
# CELL 7 -- Verify both activity types exist
n     = cnt(f"{sv}.mk_account_events_mapped")
anon  = spark.sql(f"SELECT COUNT(*) FROM {sv}.mk_account_events_mapped WHERE anonymous_activity = true").collect()[0][0]
ident = spark.sql(f"SELECT COUNT(*) FROM {sv}.mk_account_events_mapped WHERE anonymous_activity = false").collect()[0][0]
print(f"  mk_account_events_mapped: {n:,} rows")
print(f"  Identified events: {ident:,}")
print(f"  Anonymous events : {anon:,}")
status = '✅' if anon > 0 and ident > 0 else '⚠ missing one type'
print(f"  Both types present: {status}")
dbutils.notebook.exit("Success")